# Parameter-Efficient Mitigation of Catastrophic Forgetting in Small LLMs

**Paper:** *An Empirical Comparison of Parameter-Efficient Mitigation Strategies for
Catastrophic Forgetting in Small LLMs under Compute-Constrained Continual Fine-Tuning*

**Author:** Vidit Sharma

## Instructions

1. Make sure **Runtime → Change runtime type → GPU (T4)** is selected.
2. Run cells top-to-bottom.
3. Cell 2 installs all dependencies.
4. Cell 3 (optional) mounts Google Drive to persist results across sessions.
5. Cell 4 runs the main experiment sweep for the Qwen2.5-1.5B-Instruct model.
6. Cell 5 loads the saved JSON results and reproduces the three paper figures.

**Expected runtime:** ~TODO hours on a Colab T4 for all 6 methods × 1 seed.

In [ ]:
import os, subprocess

if not os.path.isdir('Rpaper'):
    subprocess.run(['git', 'clone', 'https://github.com/vidit19sharma/Rpaper.git'], check=True)

%cd Rpaper/code
!pip install -q -r requirements.txt

In [ ]:
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/Rpaper/results'
else:
    OUTPUT_DIR = '../results'

print(f'Results will be saved to: {OUTPUT_DIR}')

In [ ]:
!python run_experiments.py \
    --model qwen \
    --methods full_ft,lora,lora_ewc,lora_replay,olora,mmr \
    --seeds 42 \
    --output_dir {OUTPUT_DIR} \
    --n_train 2000 \
    --epochs 2

In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

RESULTS_DIR = OUTPUT_DIR
TASK_SEQUENCE = ['sst2', 'mrpc', 'cola', 'mnli']
METHODS = ['full_ft', 'lora', 'lora_ewc', 'lora_replay', 'olora', 'mmr']
METHOD_LABELS = ['Full-FT', 'Vanilla LoRA', 'LoRA+EWC', 'LoRA+Replay', 'O-LoRA', 'MMR (ours)']
SEED = 42
MODEL = 'qwen'

results = {}
for method in METHODS:
    path = os.path.join(RESULTS_DIR, f'{MODEL}_{method}_{SEED}.json')
    if os.path.exists(path):
        with open(path) as f:
            results[method] = json.load(f)
    else:
        print(f'Missing: {path}')

colors = cm.tab10(np.linspace(0, 1, len(METHODS)))

# --- Figure 1: Forgetting curves -------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=False)
for ax, (task_idx, task_name) in zip(axes, enumerate(TASK_SEQUENCE)):
    for (method, label, color) in zip(METHODS, METHOD_LABELS, colors):
        if method not in results:
            continue
        R = np.array(results[method]['R_matrix'])
        x = list(range(task_idx, len(TASK_SEQUENCE)))
        y = [R[i, task_idx] for i in x]
        ax.plot(x, y, marker='o', label=label, color=color)
    ax.set_title(task_name.upper())
    ax.set_xlabel('Training stage')
    ax.set_ylabel('Metric')
    ax.set_xticks(range(len(TASK_SEQUENCE)))
    ax.set_xticklabels([t.upper() for t in TASK_SEQUENCE], rotation=30, ha='right', fontsize=7)
    ax.grid(True, alpha=0.3)
handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.08), fontsize=8)
plt.tight_layout()
plt.savefig('../paper/figures/forgetting_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved forgetting_curves.pdf')

# --- Figure 2: Compute-accuracy Pareto -------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))
for method, label, color in zip(METHODS, METHOD_LABELS, colors):
    if method not in results:
        continue
    r = results[method]
    aa = r['average_accuracy']
    t = r['total_time_s'] / 3600
    ax.scatter(t, aa, color=color, s=80, zorder=5)
    ax.annotate(label, (t, aa), textcoords='offset points', xytext=(6, 3), fontsize=8)
ax.set_xlabel('Total training time (hours)')
ax.set_ylabel('Average Accuracy (AA)')
ax.set_title('Compute–Accuracy Pareto')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../paper/figures/pareto.pdf', bbox_inches='tight')
plt.show()
print('Saved pareto.pdf')

# --- Figure 3: MMR sensitivity to replay ratio f ---------------------------
MMR_SENSITIVITY_FRACTIONS = [0.02, 0.05, 0.10, 0.20]
mmr_aa_values = []
for f in MMR_SENSITIVITY_FRACTIONS:
    tag = f'mmr_f{str(f).replace(".", "p")}'
    path = os.path.join(RESULTS_DIR, f'{MODEL}_{tag}_{SEED}.json')
    if os.path.exists(path):
        with open(path) as fh:
            d = json.load(fh)
        mmr_aa_values.append(d['average_accuracy'])
    else:
        mmr_aa_values.append(None)
        print(f'Missing sensitivity run for f={f}')

fig, ax = plt.subplots(figsize=(5, 3.5))
valid = [(f, v) for f, v in zip(MMR_SENSITIVITY_FRACTIONS, mmr_aa_values) if v is not None]
if valid:
    fs, vs = zip(*valid)
    ax.plot(fs, vs, marker='o', color='steelblue')
    ax.set_xlabel('Replay ratio $f$')
    ax.set_ylabel('Average Accuracy (AA)')
    ax.set_title('MMR Sensitivity to Replay Ratio')
    ax.set_xticks(MMR_SENSITIVITY_FRACTIONS)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No sensitivity results found', ha='center', va='center',
            transform=ax.transAxes)
plt.tight_layout()
plt.savefig('../paper/figures/mmr_sensitivity.pdf', bbox_inches='tight')
plt.show()
print('Saved mmr_sensitivity.pdf')